# Stage A — the encoder supervises itself through its own mask

One runtime, one **Run all**: download unlabelled aerial imagery, pretrain
EdgeTAM's image encoder, and leave an ordinary EdgeTAM checkpoint on your Drive
that `tools/train_encoder.py --base` accepts unchanged.

Runs the **mask arm**: hide part of the frame, ask the encoder to produce the features it would have produced had it seen the whole thing, and let the *model choose what to hide* from its own feature map. No teacher, no registered pair, no label — which is the point, because it means the thermal-only sets that the distillation arm cannot use are this arm's natural food. It is the long shot of the four and the notebook says so out loud; run it because the alternative is a report that says "we did not try".

| | |
|---|---|
| **in** | nothing staged by hand — the download cell fetches everything |
| **out** | `edgetam_stage_a_mask_512.pt` + a `stage_a.json` manifest, on your Drive |
| **needs** | a CUDA GPU and ~6 GB of free disk at the default corpora (no Hugging Face token — this arm downloads no teacher) |
| **takes** | ~10 min to download and 40–90 min to train at the default 4 000 steps; the mask preview in cell 13 costs seconds and is worth reading first |

---

## Stage A is a choice between four arms, and this notebook runs one of them

| `ARM` | what supervises the encoder | which data it can eat |
|---|---|---|
| `none` | nothing — the stock EdgeTAM weights, copied | none |
| `mask` | the encoder's own features on the unmasked frame | **anything**; one modality is enough |
| `distil` | a frozen teacher's features on a paired input | needs a teacher-side image |
| `both` | `distil` first, then `mask` on top | whatever `distil` can eat |

`ARM` is a setting, so either notebook can run any of the four. **All four
write the same file**: an ordinary EdgeTAM state dict plus a manifest naming
what produced it. That is the whole architecture of this stage — one artifact,
four ways of making it, and a single line to feed it forward:

```
python tools/train_encoder.py --base <this notebook's checkpoint> ...
```

`none` writes one too, and that matters more than it sounds. **A baseline
reached by a different code path is a baseline that drifts** — stage B loading
stock weights from one place and pretrained weights from another will
eventually differ in something nobody meant to change. So the baseline is a
file, in the same folder, with the same manifest beside it.

## The number this notebook produces is not the number that decides anything

Every arm ends with a loss. That loss is a *proxy*, and two arms' proxies do
not even share a scale — a cosine distance to a foundation model's features and
a masked-position cosine to the model's own are different quantities. What
decides whether stage A was worth its GPU hours is **stage B run from this
checkpoint against stage B run from the `none` arm's**, and nothing here can
substitute for it.

That warning is not boilerplate. On this project's model scale (a 4.8 M
encoder) and close to its domain (aerial thermal segmentation), the Caltech
RGB-T paper measured: scratch 0.687, ordinary **ImageNet 0.725**, and
self-supervised pretraining on 40 k thermal images 0.714. Ordinary RGB
pretraining beat the domain-specific self-supervision. Run `none` first.


In [ ]:
# --- Runtime, repo, GPU -------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/pretrain-notebook-sota-ndc37w"

if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# Drop this repo's modules so the fast-forward above actually takes effect:
# `git pull` changes files on disk, not modules Python already imported.
for _stale in [n for n in list(sys.modules) if n.split(".")[0] in ("src", "tools")]:
    del sys.modules[_stale]

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /content | tail -1

# --- Is this notebook the one the repo expects? -------------------------
# The repo just fast-forwarded itself; the .ipynb is a file you uploaded, and
# the two drift apart silently. This says so at cell 1 instead of an hour in.
import json as _json
NOTEBOOK = "15_pretrain_automask.ipynb"
STAMP    = "6fd910562b"
_stamps  = REPO / "notebooks" / ".stamps.json"
_want    = _json.loads(_stamps.read_text()).get(NOTEBOOK) if _stamps.is_file() else None
if _want and _want != STAMP:
    print("\n" + "=" * 74)
    print(f"!!  STALE NOTEBOOK -- this file is build {STAMP}, the repo ships {_want}")
    print( "!!  Every cell below is the old version. Re-download and re-upload:")
    print(f"!!    https://github.com/yigitkayabagci/sam-dedection/raw/{BRANCH}/notebooks/{NOTEBOOK}")
    print("=" * 74 + "\n")
else:
    print(f"notebook build {STAMP} matches the repo")


In [ ]:
# --- Does this torch actually have kernels for this GPU? ----------------
# Asked by launching one, not by reading an arch list: on a card newer than
# the torch build everything imports and the first real matmul dies mid-run.
import torch

print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"{torch.cuda.get_device_name(0)}  sm_{major}{minor}  "
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
    probe = torch.randn(256, 256, device="cuda")
    (probe @ probe).sum().item()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        (probe @ probe).sum().item()
    print("a real matmul ran, in float32 and bfloat16 -- this GPU is usable")
    del probe
else:
    print("!! no CUDA device -- nothing below will train")


In [ ]:
# --- Dependencies -------------------------------------------------------
# EdgeTAM *is* installed here, unlike the mask-pool notebooks: this stage
# trains EdgeTAM's own image encoder, so the package has to be importable.
# EdgeTAM installs itself as `sam2`, so Meta's SAM 2 must never share this
# runtime; the teacher (when there is one) runs through transformers, which
# carries an independent implementation under a different name.
before = torch.__version__
!bash scripts/setup_edgetam.sh 2>&1 | tail -5
!pip install -q -r requirements.txt
!pip install -q "transformers>=4.56" hf_transfer tqdm

# The one thing an install here must not do: replace the preinstalled torch.
# Read the version off *disk* -- reloading torch always raises, and pip cannot
# change the torch already live in this kernel. Only a restart can, which is
# exactly the window this check exists to catch.
import importlib.metadata as _md
installed = _md.version("torch")
assert installed == before, (
    f"pip replaced torch {before} with {installed} on disk. This kernel is "
    f"still running {before}, so nothing has broken yet -- restore it *before* "
    f"restarting:  !pip install -q --force-reinstall torch=={before}")

# An editable install is a .pth file that site.py reads at interpreter startup
# only, so a package installed into a running kernel never joins sys.path.
# Pointing at the checkout fixes it now rather than costing a restart.
EDGETAM = REPO / "third_party" / "EdgeTAM"
if str(EDGETAM) not in sys.path:
    sys.path.append(str(EDGETAM))
BASE_CKPT = EDGETAM / "checkpoints" / "edgetam.pt"
assert BASE_CKPT.is_file(), \
    "edgetam.pt did not download -- rerun scripts/setup_edgetam.sh and read its output"

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import transformers
print(f"transformers {transformers.__version__}, EdgeTAM at {EDGETAM}")

# The contracts everything below depends on, tested with no GPU and no data.
# If these fail, nothing after this point is worth running.
!python -m unittest -q tests.test_pretrain tests.test_automask 2>&1 | tail -3


In [ ]:
# --- Drive: where the checkpoint and its manifest land ------------------
# Training reads a few hundred thousand small JPEGs in random order and the
# Drive FUSE mount serves those an order of magnitude slower than the GPU
# consumes them, so the *data* stays on local disk. Drive holds the output,
# which is megabytes.
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/edgetam-pretrain")
DRIVE.mkdir(parents=True, exist_ok=True)
print("output root ->", DRIVE, "(the per-run folder is named in the settings cell)")


## Settings

### The corpora, and why a corpus is one line

`CORPORA` is the list this stage trains on and it is meant to be **edited**.
A row is a root, a modality and either a spec name that `aerial.SPECS` already
knows or a pair of globs spelled out. Nothing else has to be touched when you
add a dataset: the crop is derived from the pictures actually on disk, the
pairing key is the one `list_pairs` already gets right, and the index cell
prints what it found per corpus — including a loud `!!` for a glob that matched
nothing, which is the way this goes wrong.

Three modalities, three routes, and the routes are the substance. The
table below is for a **thermal** student, which is what `STUDENT`
defaults to; set `STUDENT = "rgb"` and every route becomes `rgb->rgb` —
teacher and student on the identical view, which is the canonical
published feature-distillation recipe — while the thermal-only corpora
are dropped, because an encoder that deploys on colour has no use for a
pretext task solved on radiometry it will never see.


| modality | teacher sees | student sees | what it is |
|---|---|---|---|
| `paired` | the RGB half | the registered thermal half | the published route, and the only one with real cross-modal grounding |
| `rgb` | the colour image | **its own luminance** | a proxy: same one-channel-in shape, wrong sensor. This is what lets an aerial RGB set with no thermal twin contribute |
| `thermal` | the thermal frame | the same thermal frame | an RGB teacher off its home modality — worth what cell 13 says it is worth |

Feeding the student full colour on an `rgb` corpus would teach it to use a
channel the sensor does not have, and the deployed encoder has never seen
colour in its life. Graying the student's half instead keeps the structure of
the real task.

### What may be fed in, and the one thing that may not

This stage reads **no labels at all**, so anything the pipeline can decode is
fair game — including the sets stage B trains on. That is not a loophole, it is
how SAM 2's own recipe works.

The one exception is the set stage B **grades on**. This repo's rule is that
reconstructed sets train and real-mask sets grade (`aerial.Source.role`), and
the set that grades is **VTUAV VIS** — the only one shipping real instance
masks. Pretraining on its frames and then reporting stage B's held-out number
is transductive: no label leaks, but the encoder has seen the test scenes, and
the measured size of that effect in the literature is not small. Everything
else — DroneVehicle, VisDrone, HIT-UAV, SegFly, Kust4K, RGBT234, LasHeR,
Anti-UAV410 — is free.

Two smaller things. `limit=` samples **across** a corpus rather than truncating
it, which matters because most of these sets come from video and the first
5 000 frames of VTUAV are two flights. And mix the corpora: AnyThermal reports
a variant distilled *only* on aerial data coming out **worse than the
undistilled RGB model** on urban scenes.

### The schedule

`STEPS` x `BATCH` is what actually sets the length; `EPOCHS` reshuffles. One
pass over the default corpora at batch 8 is a few thousand steps, so 4 000 is
roughly one pass — and a *starved* stage A is the failure this project has
already seen once: at 600 steps and 20 000 pairs, notebook 08's distilled arm
came out **below** the run that skipped stage A entirely.


In [ ]:
# --- Shared settings ----------------------------------------------------
from src.training.pretrain import Corpus

DATA_ROOT = Path("/content/data")
WORK      = Path("/content/work"); WORK.mkdir(parents=True, exist_ok=True)

STUDENT = "thermal"  # thermal | rgb -- WHICH encoder this run pretrains.
                     # Two deployed models are two pretrainings, and the
                     # only difference between them is what the student's
                     # half is: "rgb" reroutes every corpus to
                     # same-modality distillation and drops the
                     # thermal-only ones.
SIZE   = 512        # the student's input, and the resolution it deploys at
BATCH  = 8
STEPS  = 4000       # batches per epoch
EPOCHS = 1
SEED   = 0
DEVICE = "cuda"
LIMIT  = None       # cap per corpus, spread across it rather than truncated
PROBE_SAMPLES = 256 # windows the modality probe reads
TRUNK_LR = 5e-5
NECK_LR  = 1e-4

# --- The corpora --------------------------------------------------------
# Add a row as you find data. `spec=` borrows an existing DatasetSpec's globs
# and border; give `thermal=`/`rgb=` instead for a set nobody has written one
# for. `limit=` caps a corpus so a huge one cannot drown the others.
CORPORA = [
    # Registered RGB-T. 640x512 once the 100 px white band is cropped -- this
    # project's native resolution, so no resampling at all -- day and night.
    Corpus("dronevehicle", DATA_ROOT / "DroneVehicle", "paired",
           spec="dronevehicle", limit=20000),
    # Registered RGB-T, 4 024 pairs at 640x512, with drawn masks. Small, and
    # the probe's home: its halves are the cleanest registration on the list.
    Corpus("kust4k", DATA_ROOT / "Kust4K", "paired", spec="kust4k"),
    # Aerial RGB, no thermal twin. The proxy route, and the diversity that
    # keeps a narrow distillation from damaging what it does not cover.
    Corpus("visdrone", DATA_ROOT / "VisDrone", "rgb",
           rgb="**/images/*.jpg", limit=6000),
    # Thermal only, aerial, 80-130 m. The mask arm's natural food, and the
    # corpus whose usefulness to the teacher arm cell 13 decides.
    Corpus("hituav", DATA_ROOT / "HIT_UAV", "thermal",
           thermal="**/normal_json/**/*.jpg"),
]
# Named after the student and the arm, both of which vary across runs, so
# two runs cannot overwrite each other's checkpoint on the same Drive.
OUT = WORK / f"edgetam_stage_a_{STUDENT}_mask_{SIZE}.pt"
MIRROR = DRIVE / f"{STUDENT}-mask"
MIRROR.mkdir(parents=True, exist_ok=True)
print(f"{len(CORPORA)} corpora configured -> {OUT.name}")
print(f"output -> {MIRROR}")


In [ ]:
# --- This arm's settings ------------------------------------------------
ARM        = "mask"          # none | mask | distil | both
MASK_MODE  = "hint"          # random | attentive | hint | block | green
MASK_RATIO = 0.5             # fraction of the 32x32 feature grid hidden
MASK_HINT  = 0.1             # of the masked cells, the most salient tenth
                             # stays visible (AttMask's remedy for a task
                             # that becomes unsolvable rather than hard)
SALIENCY   = "distinct"      # distinct | norm -- how a cell's interest is
                             # scored, given a conv trunk has no attention
TARGET     = "ema"           # ema | frozen -- what the masked student aims
                             # at. `frozen` cannot collapse and cannot beat
                             # the stock encoder; `ema` can do both.
TARGET_EMA = 0.999


In [ ]:
# --- No gated download in this arm --------------------------------------
# The mask arm's supervision is the encoder's own feature map, so nothing is
# fetched from a gated repository and no Hugging Face account is needed. The
# teacher notebook logs in here instead.
print("mask arm: no teacher, no Hugging Face token required")


## The data

Every URL is baked into `tools/fetch_datasets.py` and was checked against the
live host, because each of these sets is served in a way that defeats the
obvious approach. Nothing here is staged by hand.

Stage A reads **no labels at all**, which is why these sets carry far more
usable frames than annotated ones — and why a set whose annotations are the
wrong shape for segmentation (DroneVehicle's oriented boxes) is perfectly good
here.


In [ ]:
# --- Download -----------------------------------------------------------
# (name, destination, parts). A part list of [] takes the recipe's defaults.
FETCH = [
    ("dronevehicle", DATA_ROOT / "DroneVehicle", ["train"]),
    ("kust4k",       DATA_ROOT / "Kust4K",       ["tir", "rgb"]),
    ("visdrone",     DATA_ROOT / "VisDrone",     ["train"]),
    ("hituav",       DATA_ROOT / "HIT_UAV",      []),
]

for name, dest, parts in FETCH:
    have = dest.exists() and any(p.suffix.lower() in (".jpg", ".png", ".jpeg")
                                 for p in dest.rglob("*") if p.is_file())
    if have:
        print(f"{name}: already on disk at {dest}")
        continue
    extra = " --parts " + " ".join(parts) if parts else ""
    !python tools/fetch_datasets.py {name} --dest {dest}{extra} 2>&1 | tail -4

!df -h /content | tail -1


In [ ]:
# --- What will actually train -------------------------------------------
# Read off disk, not off the settings: a glob that matched nothing is the way
# this goes wrong, and it is loud here rather than silent an hour later.
from src.training import pretrain

CORPORA = pretrain.for_student(CORPORA, STUDENT)
ITEMS = pretrain.index(CORPORA, size=SIZE, seed=SEED)
print()
print(pretrain.summarise(ITEMS))
print()
print(f"one pass at batch {BATCH} is {len(ITEMS) // BATCH:,} steps; "
      f"STEPS={STEPS} and EPOCHS={EPOCHS} asks for "
      f"{STEPS * EPOCHS:,} ({STEPS * EPOCHS * BATCH / max(len(ITEMS), 1):.2f} passes)")


## Look at the mask before paying for it

The one thing that can go wrong here without raising anything: a
saliency-driven mask that hides the wrong half of the picture. On aerial
thermal the frame is mostly uniform ground with a few small hot targets, and
"mask the most distinctive cells" and "mask every target and leave the ground"
are the same instruction. That is either exactly the right pretext task or a
way to spend every gradient on 2 % of the frame.

The cell below draws it: the frame, the model's own saliency, and what
`MASK_MODE` actually hides — for the configured mode and for `random` and
`green` beside it, since those are the two baselines this arm has to beat.
Read it, then decide whether to run.


In [ ]:
# --- What does the model choose to hide? --------------------------------
import matplotlib.pyplot as plt
import torch

from src.training import automask, pretrain
from src.training.distill import encoder_features
from tools.train_encoder import build_model

preview_model = build_model(SIZE, BASE_CKPT, DEVICE)
shown = pretrain.subsample(ITEMS, 4, seed=SEED)
batch = pretrain.collate(shown, size=SIZE, device=DEVICE, want_teacher=False)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16,
                                     enabled=DEVICE.startswith("cuda")):
    clean = encoder_features(preview_model, batch.student)
scores = automask.saliency(clean, SALIENCY)

modes = [MASK_MODE] + [m for m in ("random", "green") if m != MASK_MODE]
figure, axes = plt.subplots(len(shown), 2 + len(modes),
                            figsize=(3 * (2 + len(modes)), 3 * len(shown)))
axes = axes.reshape(len(shown), -1)
for row, item in enumerate(shown):
    picture = pretrain.to_pixels(batch.student[row:row + 1])[0].permute(1, 2, 0)
    axes[row, 0].imshow(picture.clamp(0, 1).float().cpu())
    axes[row, 0].set_title(item.corpus if row == 0 else "", fontsize=9)
    axes[row, 1].imshow(scores[row].float().cpu(), cmap="magma")
    axes[row, 1].set_title("saliency" if row == 0 else "", fontsize=9)
    for column, mode in enumerate(modes):
        hidden = automask.sample_mask(scores, ratio=MASK_RATIO, mode=mode,
                                      hint=MASK_HINT)
        blanked = automask.blank(batch.student, hidden)
        axes[row, 2 + column].imshow(
            pretrain.to_pixels(blanked[row:row + 1])[0]
            .permute(1, 2, 0).clamp(0, 1).float().cpu())
        axes[row, 2 + column].set_title(mode if row == 0 else "", fontsize=9)
for axis in axes.ravel():
    axis.axis("off")
plt.tight_layout(); plt.show()

del preview_model, clean, scores
import gc; gc.collect()
if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache()
print("If the configured mode hides only the targets and leaves the ground, "
      "lower MASK_RATIO or switch to 'green' -- a mask that removes every "
      "object is not a hard task, it is an impossible one.")


## Train

Everything below goes through `tools/pretrain_stage_a.py`, the same entry point
every arm uses, so the freeze, the parameter groups, the one-cycle schedule,
the EMA and the checkpoint format are identical across arms by construction —
which is the only reason two arms' stage-B numbers can be set beside each
other.

Only the trunk and the neck move. The mask decoder, the prompt encoder and the
whole memory path stay frozen: stage A is an encoder stage, and the memory path
is the write port of a recurrent loop where error accumulates.


In [ ]:
# --- Run the arm --------------------------------------------------------
ARGS = (f"--arm {ARM} --mask-mode {MASK_MODE} --mask-ratio {MASK_RATIO} "
        f"--mask-hint {MASK_HINT} --saliency {SALIENCY} "
        f"--target {TARGET} --target-decay {TARGET_EMA}")

CORPORA_JSON = WORK / "corpora.json"
import json as _cjson
CORPORA_JSON.write_text(_cjson.dumps(
    [{"name": c.name, "root": str(c.root), "modality": c.modality,
      "spec": c.spec, "thermal": c.thermal, "rgb": c.rgb, "route": c.route,
      "limit": c.limit, "border": c.border, "exclude": c.exclude,
      "crop": c.crop} for c in CORPORA], indent=2))

REPORT_JSON = WORK / "stage_a_report.json"
!python tools/pretrain_stage_a.py {ARGS} \
    --corpora {CORPORA_JSON} --base {BASE_CKPT} --out {OUT} \
    --student {STUDENT} --size {SIZE} --batch {BATCH} --steps {STEPS} \
    --epochs {EPOCHS} \
    --trunk-lr {TRUNK_LR} --neck-lr {NECK_LR} --seed {SEED} \
    --device {DEVICE} --json {REPORT_JSON}

RESULT = _cjson.loads(REPORT_JSON.read_text())
print(f"\n{RESULT['arm']}: {RESULT['steps']} steps over {RESULT['samples']:,} "
      f"samples in {RESULT['seconds'] / 60:.0f} min")


In [ ]:
# --- Did the representation survive? ------------------------------------
# The failure this checks for is the one a loss curve hides completely: an EMA
# self-distillation that collapses has a loss falling beautifully toward zero
# while every position in the map becomes the same vector.
from src.training.automask import summarise_collapse

print(summarise_collapse(RESULT["history"]))


In [ ]:
# --- Stage it, with the manifest beside it ------------------------------
# The manifest is the point: stage B takes a path and asks no questions, so
# without it the record of which arm, which corpora and which seed produced a
# checkpoint lives in a Colab cell that scrolls away.
import shutil

from src.training.pretrain import read_manifest, stage_b_command

STAGED = MIRROR / OUT.name
shutil.copy(OUT, STAGED)
MANIFEST = OUT.with_suffix(".stage_a.json")
if MANIFEST.is_file():
    shutil.copy(MANIFEST, MIRROR / MANIFEST.name)
shutil.copy(REPORT_JSON, MIRROR / REPORT_JSON.name)

print(f"{STAGED}  ({STAGED.stat().st_size / 2**20:.0f} MiB)")
for key, value in sorted(read_manifest(OUT).items()):
    if key != "history":
        print(f"  {key:<14} {value}")

print()
print("Feed it to stage B with:")
print(stage_b_command(STAGED))


## Handing this to stage B

The checkpoint is an **ordinary EdgeTAM state dict** — same keys, same loader,
same exporter — with a differently-trained trunk and neck inside it. Nothing
downstream needs to know this stage happened, which is exactly what makes the
comparison possible:

```
# the arm you just ran
python tools/train_encoder.py --base <MIRROR>/edgetam_stage_a_mask_512.pt \
    --out checkpoints/stage_b_mask.pt --dataset ...

# the baseline it has to beat, same everything else
python tools/pretrain_stage_a.py --arm none --out work/stage_a_none.pt
python tools/train_encoder.py --base work/stage_a_none.pt \
    --out checkpoints/stage_b_none.pt --dataset ...
```

Notebook `07_encoder_aerial_rgbt.ipynb` is stage B; it runs its own inline
stage A today, so point its `--base` at this file to use this one instead. Hold
**everything else** fixed — the datasets, the schedule, the seed, the prompt
mix — or the two `test/instance_iou` numbers are not comparable and the
comparison was the only reason to run this.

### What a result looks like

Four numbers, from four stage-B runs that differ in one flag:

| stage A | notebook | stage B `test/instance_iou` |
|---|---|---|
| `none` | any, `--arm none` | — |
| `mask` | 15 | — |
| `distil`, ViT-B/16 | 16 | — |
| `distil`, ConvNeXt-S | 17 | — |

If an arm does not beat `none`, it cost GPU hours and bought nothing, and that
is a finding worth writing down rather than a run worth hiding. The literature
is full of exactly that outcome at this model scale.

---

## What this arm is, and what to expect from it

**The method, in one line.** A copy of the encoder sees the clean frame and
produces the target map; the encoder being trained sees the frame with half of
its 16-pixel cells blanked to the mean pixel and must reproduce that map at the
blanked positions. The mask is chosen from the target's own feature map — the
cells whose features are furthest from the frame's average direction — which is
the convolutional stand-in for AttMask's "mask what the class token attends
to", since a RepViT trunk has no attention map to read.

**Why there is no sparse convolution.** ConvNeXt V2's FCMAE needed it because a
dense convolution over a masked image leaks: the kernel straddles the boundary
and the hole's shape propagates, so the network can learn *where* the mask was.
That argument bites hardest when the target is pixels, because a model that
knows where the hole is can hedge. Here the target is a feature map and there
is no decoder, so the leak buys much less — and the cost of the alternative is
re-implementing a trunk this repo does not own.

**But the leak is not zero, and one part of it is named.** RepViT's later
stages carry squeeze-and-excitation blocks, and an SE gate is a global average
over the map. Averaged over a masked input that gate is biased by the visible
fraction, and at stage B there is no mask at all. The mitigations here are the
cheap ones — the fill is the mean pixel, which is the least informative value
available, and `MASK_RATIO` defaults to 0.5 rather than MAE's 0.75. If this arm
wins, patching the pooling to count only visible positions (what SparK does) is
the first thing to try next.

## The honest prior

Four measured numbers, none of them ours, all pointing the same way:

| what was measured | result |
|---|---|
| ConvNeXt V2's FCMAE on an **unmodified** architecture (its own Table 14) | −0.1 to +0.1 top-1. The headline +1.0 came from co-designing the GRN layer alongside it |
| MAE pretraining a **5.7 M** ViT (TinyMIM) | **−0.6** against training from scratch |
| a learned mask against a random one, everything else fixed (HPM's isolation ablation) | +0.46; and +0.26 of its +0.72 was simply adding the auxiliary predictor |
| aerial thermal segmentation at **4.8 M** parameters (Caltech RGB-T, ECCV 2024) | scratch 0.687, **ImageNet 0.725**, self-supervision on 40 k thermal images 0.714 |

The last row is the one to sit with: on this project's model scale and close to
its domain, ordinary RGB pretraining beat thermal self-supervision. That does
not make this arm pointless — it makes it a *measurement*, and the thing it
measures is whether a thermal-only corpus in the deployment's own modality can
add anything the other arm structurally cannot reach.

`MASK_MODE = "green"` is the setting worth respecting here. It is ColorMAE
(ECCV 2024): band-pass-filtered noise, no parameters, no extra forward pass,
and in its own controlled comparison it matched the learned maskers of the
three years before it. If `hint` does not beat `green` on stage B's number,
the self-generated mask earned nothing and the honest thing is to say so.

## Reading the collapse table

`similarity` is the mean cosine between two *different* positions in the same
frame. A dense map is useful precisely because its positions disagree; a number
approaching 1 means they have stopped, and the checkpoint is then worse than
doing nothing whatever the loss says. `channel_std` is the same failure seen
from the other side. `visible` is a sanity check rather than an objective — the
student can see those pixels, so it should sit near 1 and stay there.

If it collapses: lower `MASK_RATIO`, raise `TARGET_EMA` toward 0.9995, or set
`TARGET = "frozen"`, which cannot collapse because the target never moves.
